# How the analyst builds a memo, step by step

[`financial-analyst-agent`](https://github.com/godot107/financial-analyst-agent) answers a question
about a public company from its 10-K. **Claude chooses the ratios and writes the prose; Python
computes every number**, and a checker rejects any digit the model typed itself.

This notebook opens up each step: the figures pulled from the filing, the ratios built from them,
the paragraphs retrieved to explain a movement, the checks a draft has to pass, and the trace of a
whole run.

**It costs nothing to run.** Every cell below reads a recorded filing from `tests/fixtures/` and a
draft recorded from a real run, so there is no network call, no API key and no spend. The last
section runs it live against Claude, and is off by default.

In [1]:
import io
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
pd.set_option("display.max_colwidth", 60)

FIXTURES = ROOT / "tests" / "fixtures"
MILLIONS = 1e6

## 1. What comes out of the filing

`edgar.py` pulls a fixed list of line items from the 10-K's XBRL data. Every fact records the tag it
matched, the period it belongs to, and the filing it came from, so any figure in the memo can be
traced back to a line in a filing.

Note `reported`: a line the company genuinely doesn't have (Microsoft has no commercial paper)
counts as 0 and is labelled, rather than being confused with a tag we failed to recognise.

In [2]:
from fin_analyst.edgar import load_facts

facts = load_facts(FIXTURES / "msft_facts.json")
latest = max(f.fiscal_year for f in facts if f.line_item == "total_assets")

pd.DataFrame(
    [
        {"line_item": f.line_item, "year": f.fiscal_year, "$m": f.value / MILLIONS,
         "tag": f.concept, "period": f.period, "reported": f.reported}
        for f in facts if f.fiscal_year == latest
    ]
).sort_values("line_item").reset_index(drop=True)

,line_item,year,$m,tag,period,reported
0,accounts_receivable,2026,80876.0,us-gaap:AccountsReceivableNetCurrent,2026-06-30,True
1,cash,2026,20935.0,us-gaap:CashAndCashEquivalentsAtCarryingValue,2026-06-30,True
2,current_assets,2026,207710.0,us-gaap:AssetsCurrent,2026-06-30,True
3,current_liabilities,2026,168825.0,us-gaap:LiabilitiesCurrent,2026-06-30,True
4,current_long_term_debt,2026,9227.0,us-gaap:LongTermDebtCurrent,2026-06-30,True
5,diluted_shares,2026,7453.0,us-gaap:WeightedAverageNumberOfDilutedSharesOutstanding,2026-06-30,True
6,equity,2026,442387.0,us-gaap:StockholdersEquity,2026-06-30,True
7,finance_lease_liabilities,2026,66594.0,us-gaap:FinanceLeaseLiability,2026-06-30,True
8,gross_profit,2026,225465.0,us-gaap:GrossProfit,2026-06-30,True
9,interest_expense,2026,3051.0,us-gaap:InterestExpenseNonoperating,2026-06-30,True


## 2. The ratios

Each ratio is a small Python function with a textbook definition attached (Berk & DeMarzo,
Subramanyam). It records the figures it used, so the arithmetic can be checked without rerunning
anything. Where an input is missing, the result carries a reason instead of a number — never a
guess.

In [3]:
from fin_analyst.metrics import compute_all
from fin_analyst.memo import format_value

wanted = ["current_ratio", "quick_ratio", "cash_flow_ratio", "debt_to_equity",
          "debt_to_equity_with_leases", "interest_coverage", "roe_average_equity"]
results = compute_all(facts, wanted)

pd.DataFrame(
    [
        {"metric": r.metric_id, "year": r.fiscal_year, "value": format_value(r),
         "built from": ", ".join(f"{k} {v / MILLIONS:,.0f}" for k, v in r.inputs.items()) or r.reason}
        for r in results if r.fiscal_year >= latest - 1
    ]
)

,metric,year,value,built from
0,current_ratio,2026,1.23x,"current_assets 207,710, current_liabilities 168,825"
1,current_ratio,2025,1.35x,"current_assets 191,131, current_liabilities 141,218"
2,quick_ratio,2026,0.93x,"cash 20,935, short_term_investments 55,908, accounts_rec..."
3,quick_ratio,2025,1.16x,"cash 30,242, short_term_investments 64,323, accounts_rec..."
4,cash_flow_ratio,2026,1.08x,"operating_cash_flow 182,935, current_liabilities 168,825"
5,cash_flow_ratio,2025,0.96x,"operating_cash_flow 136,162, current_liabilities 141,218"
6,debt_to_equity,2026,0.09x,"short_term_borrowings 0, current_long_term_debt 9,227, l..."
7,debt_to_equity,2025,0.13x,"short_term_borrowings 0, current_long_term_debt 2,999, l..."
8,debt_to_equity_with_leases,2026,0.29x,"short_term_borrowings 0, current_long_term_debt 9,227, l..."
9,debt_to_equity_with_leases,2025,0.33x,"short_term_borrowings 0, current_long_term_debt 2,999, l..."


## 3. Finding the paragraphs that explain a movement

Ratios say *what* moved; only the filing's narrative says *why*. Item 7 and Item 1A are split into
paragraphs and searched with BM25 — plain term matching, no vector database.

The first memo this service produced in production got this wrong, and the trace is what showed it.
Asked "How liquid is Microsoft?", the search had two words to work with after stopwords, and
**"Microsoft" did the ranking**: three of the four paragraphs it returned were about Microsoft 365
revenue. The fix was to drop the company's own name from the query, and to stem "liquidity" to
"liquid".

In [4]:
from fin_analyst.passages import load_passages, name_words, search

passages = load_passages(FIXTURES / "msft_passages.json")
question = "How liquid is Microsoft?"

def show(found):
    return [f"[{p.id}] {p.text[:95]}..." for p in found]

pd.DataFrame({
    "with the company name (what production did)": show(search(passages, question, 4)),
    "without it (what it does now)": show(search(passages, question, 4, name_words("MICROSOFT CORP"))),
})

,with the company name (what production did),without it (what it does now)
0,"[P50] Cash, cash equivalents, and short-term investments...","[P50] Cash, cash equivalents, and short-term investments..."
1,[P17] Revenue from Microsoft 365 Commercial subscription...,[P54] We issue debt to take advantage of favorable prici...
2,[P25] •Microsoft 365 Commercial products and cloud servi...,[P80] Our operations and financial results are subject t...
3,[P26] •Microsoft 365 Consumer products and cloud service...,[P159] We maintain an investment portfolio of various ho...


## 4. What a draft has to pass

Claude writes placeholders, never figures. The checker reads a draft and returns problems in the
words the writer needs to fix them; any problem sends the draft back to be rewritten, and three
failures publish nothing at all.

The draft below packs in four different problems, three of which came from real memos.

In [5]:
from fin_analyst.memo import find_problems, render

bad = (
    "Liquidity fell to 1.23x, which is healthy. "
    "The current ratio {{current_ratio:2025->2026}}, leaving it at {{current_ratio:2026}}. "
    "Margins improved [P99]."
)
for problem in find_problems(bad, results, passages=passages):
    print("-", problem)

- {{current_ratio:2025->2026}} already ends on the 2026 value, so {{current_ratio:2026}} in the same sentence says it twice; remove one
- "healthy" judges a ratio against a standard the data does not contain; describe the change against the prior year instead
- '1.23x,' is a number you wrote yourself; every number must be a placeholder


In [6]:
good = "The current ratio {{current_ratio:2025->2026}}, and the quick ratio {{quick_ratio:2025->2026}}."
print("problems:", find_problems(good, results, passages=passages))
print()
print(render(good, results))  # the renderer supplies the value *and* the direction

problems: []

The current ratio fell 0.12x to 1.23x, and the quick ratio fell 0.23x to 0.93x.


## 5. A whole run, traced

The workflow is a LangGraph graph: `plan → fetch → compute → retrieve → write → check → verify →
render`, with a retry loop back to `write`. Claude appears twice (choosing metrics, writing) plus
once more to check the citations hold.

To keep this notebook free, the analyst below is a stand-in that replays a draft recorded from a
real run. Everything else — retrieval, the checks, the claim check's bookkeeping, rendering — is the
real code, including the retrieval fix: the run is told which company filed, and the search leaves
that name out of the question. The trace below is what the service writes to CloudWatch on every
production run.

In [7]:
from fin_analyst.config import load_settings
from fin_analyst.graph import run_analysis
from fin_analyst.edgar import Filing
from fin_analyst.trace import Tracer, pretty

# What the EDGAR lookup returns for this filing. The company's name matters:
# the search leaves it out of the question, which is why the passages below are
# about liquidity rather than Microsoft 365 revenue.
MSFT_FILING = Filing(
    ticker="MSFT", company="MICROSOFT CORP", cik=789019, sic=7372, form="10-K",
    accession="0001193125-26-323660", filed="2026-07-29",
    url="https://www.sec.gov/Archives/edgar/data/789019/0001193125-26-323660-index.html",
)

RECORDED_DRAFT = "## Memo: Microsoft liquidity, FY2026 10-K\n\n**One line:** On the balance-sheet measures liquidity tightened against FY2025, while liquidity from operating cash flow improved.\n\n**What moved**\n\n- The current ratio, current assets over current liabilities \u2014 whether the next year's bills are covered by assets expected to convert within the year \u2014 {{current_ratio:2025->2026}}.\n- The quick ratio, a stricter test that counts only cash, short-term investments and receivables against current liabilities, {{quick_ratio:2025->2026}}. This is the larger of the two declines, meaning the narrower pool of liquid assets shrank relative to current liabilities faster than current assets as a whole.\n- The cash flow ratio, operating cash flow over current liabilities \u2014 liquidity from the cash the business itself generates rather than from assets on hand \u2014 {{cash_flow_ratio:2025->2026}}.\n\n**What the filing says about it**\n\nThe filing reports that cash, cash equivalents and short-term investments were lower at June 30, 2026 than a year earlier, while equity and other investments were higher [P50]; the quick ratio counts the former and not the latter, which is consistent with the direction of that measure. The filing describes short-term investments as primarily intended to facilitate liquidity and capital preservation, held predominantly in highly liquid investment-grade fixed-income securities [P50]. It also notes that debt is issued to take advantage of pricing and liquidity in the debt markets, for general corporate purposes including working capital [P54]. Beyond this, the filing does not explain the year-over-year movement in these ratios here.\n\n**Limits**\n\nThese are Microsoft against its own prior year only \u2014 there is no peer comparison in this data, and no benchmark level to judge the ratios against."

class RecordedAnalyst:
    """Stands in for Claude: the same three methods, with recorded answers."""

    spent_usd = 0.0490  # what the real run cost

    def choose_metrics(self, ticker, question):
        return ["current_ratio", "quick_ratio", "cash_flow_ratio"], 0.0

    def write_draft(self, ticker, question, metrics, problems, history=(), peer_ticker=None,
                    peer_metrics=(), passages=()):
        return RECORDED_DRAFT, 0.0

    def verify_claims(self, claims):
        from fin_analyst.llm import Verdict
        return [Verdict(supported=True, reason="the passage states it") for _ in claims], 0.0

trace_output = io.StringIO()  # collected, then printed in one go
state = run_analysis(
    "MSFT", "How liquid is Microsoft?", RecordedAnalyst(), load_settings(),
    fetch=lambda ticker: facts,
    fetch_text=lambda ticker: passages,
    describe=lambda ticker: MSFT_FILING,
    runs_dir=Path("/tmp/notebook-runs"),
    tracer=Tracer([pretty(trace_output)]),
)
print(trace_output.getvalue())

[   0.0s] run      start  ticker=MSFT  question=How liquid is Microsoft?  text=True  verify=True  news=False  market=False
[   0.0s] plan     done  seconds=0.0  metric_ids=current_ratio, quick_ratio, cash_flow_ratio
[   0.0s] fetch    done  seconds=0.0  filing=MICROSOFT CORP 10-K filed 2026-07-29 (0001193125-26-323660)  facts=48  fiscal_years=2024, 2025, 2026
[   0.0s] compute  done  seconds=0.0
           values:
             - current_ratio:2026 = 1.2303
             - current_ratio:2025 = 1.3534
             - quick_ratio:2026 = 0.9342
             - quick_ratio:2025 = 1.1647
             - cash_flow_ratio:2026 = 1.0836
             - cash_flow_ratio:2025 = 0.9642
[   0.0s] retrieve done  seconds=0.02
           passages:
             - [P50] Item 7: Cash, cash equivalents, and short-term investments totaled $76.8 billion and $94.6 billion as of June 30, 2026 and 2025, respectively. Equit...
             - [P54] Item 7: We issue debt to take advantage of favorable pricing and liquid

### The memo

Every figure below was substituted by the renderer from the facts in step 1. The sources and the
footer are added by code, not written by the model.

In [8]:
from IPython.display import Markdown

Markdown(state.memo)

## Memo: Microsoft liquidity, FY2026 10-K

**One line:** On the balance-sheet measures liquidity tightened against FY2025, while liquidity from operating cash flow improved.

**What moved**

- The current ratio, current assets over current liabilities — whether the next year's bills are covered by assets expected to convert within the year — fell 0.12x to 1.23x.
- The quick ratio, a stricter test that counts only cash, short-term investments and receivables against current liabilities, fell 0.23x to 0.93x. This is the larger of the two declines, meaning the narrower pool of liquid assets shrank relative to current liabilities faster than current assets as a whole.
- The cash flow ratio, operating cash flow over current liabilities — liquidity from the cash the business itself generates rather than from assets on hand — rose 0.12x to 1.08x.

**What the filing says about it**

The filing reports that cash, cash equivalents and short-term investments were lower at June 30, 2026 than a year earlier, while equity and other investments were higher [P50]; the quick ratio counts the former and not the latter, which is consistent with the direction of that measure. The filing describes short-term investments as primarily intended to facilitate liquidity and capital preservation, held predominantly in highly liquid investment-grade fixed-income securities [P50]. It also notes that debt is issued to take advantage of pricing and liquidity in the debt markets, for general corporate purposes including working capital [P54]. Beyond this, the filing does not explain the year-over-year movement in these ratios here.

**Limits**

These are Microsoft against its own prior year only — there is no peer comparison in this data, and no benchmark level to judge the ratios against.

**Cited sources**
- [P50] Item 7, filing 0001193125-26-323660: "Cash, cash equivalents, and short-term investments totaled $76.8 billion and $94.6 billion as of June 30, 2026 and 2025, respectively. Equity and other investments were $36.3 billion and $15.4 billion as of June 30, 2026..."
- [P54] Item 7, filing 0001193125-26-323660: "We issue debt to take advantage of favorable pricing and liquidity in the debt markets, reflecting our credit rating. The proceeds of these issuances were or will be used for general corporate purposes, which may include..."

---
MSFT: SEC filing 0001193125-26-323660; balance sheet dates 2026-06-30, 2025-06-30.
Ratios use ending balances, not averages. Debt excludes lease liabilities.
No peer comparison: this memo covers one company against its own prior years.
MSFT does not report, so treated as zero: short_term_borrowings.

## 6. The same memo as data

The deployed service returns this alongside the Markdown, so a program never has to parse prose:
the filing it used, each ratio with its inputs, and each passage marked cited or not.

In [9]:
from fin_analyst.service import structured_result

result = structured_result(state)
print({k: (f"{len(v)} items" if isinstance(v, list) else v) for k, v in result.items()
       if k not in ("trace", "facts", "peer_facts")})
print()
print(json.dumps(result["metrics"][0], indent=2))

{'filing': {'ticker': 'MSFT', 'company': 'MICROSOFT CORP', 'cik': 789019, 'sic': 7372, 'form': '10-K', 'accession': '0001193125-26-323660', 'filed': '2026-07-29', 'period': '2026-06-30', 'url': 'https://www.sec.gov/Archives/edgar/data/789019/0001193125-26-323660-index.html'}, 'peer_filing': None, 'metric_ids': '3 items', 'metrics': '6 items', 'peer_metrics': '0 items', 'passages': '4 items', 'claim_checks': '3 items', 'drafts': 1}

{
  "metric_id": "current_ratio",
  "fiscal_year": 2026,
  "value": 1.2303272619576484,
  "unit": "ratio",
  "reason": null,
  "inputs": {
    "current_assets": 207710000000.0,
    "current_liabilities": 168825000000.0
  },
  "formatted": "1.23x"
}


## 7. A company these ratios don't fit

A bank's balance sheet isn't split into current and non-current, and its debt isn't in the tags a
commercial balance sheet uses. Computing the usual ratios anyway gave JPMorgan **0.18x** debt to
equity — wrong, and plausible enough to publish. So those ratios are refused, and the ratios a bank
actually runs on are computed instead.

In [10]:
jpm = load_facts(FIXTURES / "jpm_facts.json")

pd.DataFrame(
    [
        {"metric": r.metric_id, "year": r.fiscal_year, "value": format_value(r)[:80]}
        for r in compute_all(jpm, ["current_ratio", "debt_to_equity", "interest_coverage",
                                   "efficiency_ratio", "loans_to_deposits", "credit_cost_to_loans",
                                   "roe_average_equity"])
        if r.fiscal_year >= max(f.fiscal_year for f in jpm if f.line_item == "total_assets")
    ]
)

,metric,year,value
0,current_ratio,2025,not available (doesn't apply: the balance sheet isn't sp...
1,debt_to_equity,2025,not available (doesn't apply: the balance sheet isn't sp...
2,interest_coverage,2025,not available (doesn't apply: the balance sheet isn't sp...
3,efficiency_ratio,2025,52.4%
4,loans_to_deposits,2025,0.57x
5,credit_cost_to_loans,2025,1.0%
6,roe_average_equity,2025,16.1%


## 8. Running it for real

Everything above ran on recorded data. To ask Claude a fresh question, set `LIVE = True` below. That
needs `ANTHROPIC_API_KEY` and `SEC_USER_AGENT` in `.env`, and **costs about $0.05 per memo**.

The same run is available from the command line (`python -m fin_analyst MSFT "..." --verbose`) and
from the deployed service (`./deploy/call.sh POST /v1/memos ...`, see `docs/CALLING.md`).

In [11]:
LIVE = False  # set to True to spend about $0.05

if LIVE:
    from functools import partial

    from dotenv import load_dotenv

    from fin_analyst.cache import LocalCache
    from fin_analyst.edgar import describe_filing, fetch_facts
    from fin_analyst.llm import ClaudeAnalyst
    from fin_analyst.passages import fetch_passages

    load_dotenv(ROOT / ".env")
    settings = load_settings()
    cache = LocalCache(ROOT / "cache")
    analyst = ClaudeAnalyst(settings)

    live = run_analysis(
        "COST", "How liquid is Costco?", analyst, settings,
        fetch=partial(fetch_facts, cache=cache),
        fetch_text=partial(fetch_passages, cache=cache),
        describe=describe_filing,
        runs_dir=ROOT / "runs",
        tracer=Tracer([pretty(sys.stdout)]),
    )
    print(f"\n${analyst.spent_usd:.4f}")
    display(Markdown(live.memo or f"No memo: {live.error}"))
else:
    print("LIVE is off: nothing was called and nothing was spent.")

LIVE is off: nothing was called and nothing was spent.


---

**Where to read more**

- [`README.md`](../README.md) — what it does, what it costs, and what it can't do
- [`BLOG.md`](../BLOG.md) and [`BLOG-2.md`](../BLOG-2.md) — why it's built this way, and what the
  checks and traces caught
- [`docs/CALLING.md`](../docs/CALLING.md) — calling the deployed service, and reading its traces
- [`evals/gold.py`](../evals/gold.py) — figures checked by hand against five 10-Ks

Not investment advice.